# NVC on Google Colab

Installs this project into an isolated `uv` environment, downloads **v2** assets only, then launches the Gradio GUI.

### Colab runtime used by this notebook
Current GPU image from [`googlecolab/backend-info`](https://github.com/googlecolab/backend-info):

| Item | Typical Colab GPU runtime |
| --- | --- |
| OS | Ubuntu 22.04.5 LTS |
| Python | 3.12.13 |
| CUDA toolkit | 12.8 |
| Preinstalled Torch | 2.11.0+cu128 |
| Preinstalled NumPy | 2.0.2 |
| Preinstalled Gradio | 6.x |
| Already useful | `git`, NVIDIA driver, `ffmpeg` (reinstalled below) |

NVC needs **Python 3.12**, **Torch 2.7.1+cu128** (Torch must stay below 2.8), **NumPy 1.x**, and **Gradio 3.14**. Those pins conflict with Colab system site-packages, so nothing is installed into `/usr`. All Python deps go into `/content/nvc-venv` via `uv`.

### Before you run
1. `Runtime -> Change runtime type -> GPU` (T4 / L4 / A100).
2. Set `NVC_GIT_URL` in the settings cell to your NVC repository.
3. Run the cells in order.


In [ ]:
# GPU + OS check. Stop here if this is not a CUDA runtime.
import os
import shutil
import subprocess
import sys

assert shutil.which("nvidia-smi"), "Enable a GPU runtime: Runtime -> Change runtime type -> GPU"
subprocess.run(["nvidia-smi"], check=True)
print("python:", sys.version.split()[0])
print("cuda visible:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))


## Settings


In [ ]:
# @title Settings
from pathlib import Path

NVC_GIT_URL = "https://github.com/kanoyo-git/NVC.git"  # @param {type:"string"}
NVC_GIT_REF = "main"  # @param {type:"string"}
PROJECT_DIR = "/content/NVC"  # @param {type:"string"}
VENV_DIR = "/content/nvc-venv"  # @param {type:"string"}
USE_DRIVE_CACHE = False  # @param {type:"boolean"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/NVC-cache"  # @param {type:"string"}
LAUNCH_GUI = True  # @param {type:"boolean"}
GUI_PORT = 7865  # @param {type:"integer"}

PROJECT = Path(PROJECT_DIR)
VENV = Path(VENV_DIR)
PY = VENV / "bin" / "python"
UV_BIN = None


## System packages and `uv`


In [ ]:
# ffmpeg / audio libs from apt. uv from the official standalone installer.
import os
import shutil
import subprocess
from pathlib import Path

def sh(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, **kwargs)

def resolve_uv():
    extra = [
        Path.home() / ".local" / "bin",
        Path("/root/.local/bin"),
        Path.home() / ".cargo" / "bin",
        Path("/usr/local/bin"),
    ]
    os.environ["PATH"] = os.pathsep.join(str(p) for p in extra if p.is_dir()) + os.pathsep + os.environ["PATH"]
    found = shutil.which("uv")
    if found:
        return Path(found)
    for folder in extra:
        candidate = folder / "uv"
        if candidate.is_file():
            return candidate
    return None

sh(["apt-get", "update", "-qq"])
sh([
    "apt-get", "install", "-y", "-qq",
    "ffmpeg", "libsndfile1", "libportaudio2", "git", "unzip", "curl",
])

UV_BIN = resolve_uv()
if UV_BIN is None:
    install_dir = Path.home() / ".local" / "bin"
    install_dir.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["UV_INSTALL_DIR"] = str(install_dir)
    env["UV_NO_MODIFY_PATH"] = "1"
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, env=env)
    UV_BIN = resolve_uv()
if UV_BIN is None:
    sh(["python3", "-m", "pip", "install", "-q", "uv"])
    UV_BIN = resolve_uv()
if UV_BIN is None:
    raise FileNotFoundError("uv was installed but the binary was not found on PATH")

os.environ["PATH"] = str(UV_BIN.parent) + os.pathsep + os.environ["PATH"]
sh([str(UV_BIN), "--version"])
print("uv:", UV_BIN)


## Fetch the project


In [ ]:
# Clone NVC or reuse an existing checkout. Optional Drive cache for model files.
import os
import subprocess
from pathlib import Path

def sh(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, **kwargs)

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_CACHE_DIR).mkdir(parents=True, exist_ok=True)

if (PROJECT / "gui.py").is_file():
    print("using existing project at", PROJECT)
elif NVC_GIT_URL.strip():
    if PROJECT.exists():
        sh(["rm", "-rf", str(PROJECT)])
    sh(["git", "clone", "--depth", "1", "--branch", NVC_GIT_REF, NVC_GIT_URL.strip(), str(PROJECT)])
else:
    raise SystemExit(
        "Set NVC_GIT_URL to your NVC git remote, or place the project at "
        f"{PROJECT} so that gui.py is present."
    )

os.chdir(PROJECT)
print("cwd:", Path.cwd())
print("gui.py:", (PROJECT / "gui.py").is_file())


## Python environment


In [ ]:
# Isolated uv venv. Stage 1: Torch 2.7.1+cu128. Stage 2: project requirements from official PyPI.
import os
import subprocess
from pathlib import Path

def sh(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, **kwargs)

os.chdir(PROJECT)
req_src = PROJECT / "requirments_cu128_py312.txt"
if not req_src.is_file():
    raise FileNotFoundError(req_src)

req_clean = Path("/tmp/nvc-cu128-pypi.txt")
lines = []
for line in req_src.read_text(encoding="utf-8").splitlines():
    stripped = line.strip()
    if stripped.startswith("--index-url") or stripped.startswith("--extra-index-url"):
        continue
    lines.append(line)
req_clean.write_text("\n".join(lines) + "\n", encoding="utf-8")

if not PY.is_file():
    sh([str(UV_BIN), "venv", str(VENV), "--python", "3.12", "--clear"])

sh([
    str(UV_BIN), "pip", "install", "--python", str(PY),
    "torch==2.7.1+cu128", "torchaudio==2.7.1+cu128",
    "--index-url", "https://download.pytorch.org/whl/cu128",
])
sh([
    str(UV_BIN), "pip", "install", "--python", str(PY),
    "-r", str(req_clean), "huggingface_hub>=0.25,<1",
    "--index-url", "https://pypi.org/simple",
])
sh([
    str(PY), "-c",
    "import torch; print('torch', torch.__version__); print('cuda', torch.version.cuda); print('available', torch.cuda.is_available()); print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)",
])


## v2 models and runtime assets

Downloads only:

- `assets/hubert_base/*`
- `assets/rmvpe/rmvpe.pt`
- `assets/pretrained_v2/*.pth`
- `logs/mute/*`
- `assets/pymss_weights/*`

v1 `pretrained/` weights are not downloaded.


In [ ]:
# Decode the public model repo name at runtime. Download v2 assets only.
import codecs
import os
import zipfile
from pathlib import Path

os.chdir(PROJECT)

if USE_DRIVE_CACHE:
    hf_home = Path(DRIVE_CACHE_DIR) / "huggingface"
    hf_home.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(hf_home)

from huggingface_hub import hf_hub_download, snapshot_download

MODEL_REPO = codecs.decode("yw1995/IbvprPbairefvbaJroHV", "rot_13")

for rel in (
    "assets/hubert_base",
    "assets/rmvpe",
    "assets/pretrained_v2",
    "assets/pymss_weights",
    "assets/weights",
    "assets/indices",
    "logs",
    ".model-downloads",
):
    (PROJECT / rel).mkdir(parents=True, exist_ok=True)

snapshot_download(MODEL_REPO, revision="main", allow_patterns=["hubert_base/*"], local_dir=str(PROJECT / "assets"))
hf_hub_download(MODEL_REPO, "rmvpe.pt", revision="main", local_dir=str(PROJECT / "assets" / "rmvpe"))
snapshot_download(MODEL_REPO, revision="main", allow_patterns=["pretrained_v2/*"], local_dir=str(PROJECT / "assets"))
mute_zip = hf_hub_download(MODEL_REPO, "mute.zip", revision="main", local_dir=str(PROJECT / ".model-downloads"))
snapshot_download(MODEL_REPO, revision="main", allow_patterns=["pymss_weights/*"], local_dir=str(PROJECT / "assets"))

with zipfile.ZipFile(mute_zip) as archive:
    archive.extractall(PROJECT / "logs")

required = [
    PROJECT / "assets/hubert_base/config.json",
    PROJECT / "assets/hubert_base/preprocessor_config.json",
    PROJECT / "assets/hubert_base/pytorch_model.bin",
    PROJECT / "assets/rmvpe/rmvpe.pt",
    PROJECT / "assets/pretrained_v2/f0G32k.pth",
    PROJECT / "assets/pretrained_v2/f0D32k.pth",
    PROJECT / "assets/pretrained_v2/f0G40k.pth",
    PROJECT / "assets/pretrained_v2/f0D40k.pth",
    PROJECT / "assets/pretrained_v2/f0G48k.pth",
    PROJECT / "assets/pretrained_v2/f0D48k.pth",
    PROJECT / "logs/mute",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("missing assets:\n" + "\n".join(missing))

print("v2 assets ready")
for path in sorted((PROJECT / "assets/pretrained_v2").glob("*.pth")):
    print(" ", path.name, path.stat().st_size)


## Launch


In [ ]:
# Start the Gradio GUI. --colab enables a public share URL.
import os
import subprocess
import sys

os.chdir(PROJECT)
if not LAUNCH_GUI:
    print("LAUNCH_GUI is False; environment is ready.")
else:
    env = os.environ.copy()
    env["NVC_OFFLINE_CUDA_GRAPH"] = "0"
    env["PYTHONUNBUFFERED"] = "1"
    env["MPLBACKEND"] = "Agg"
    print("Starting GUI on port", GUI_PORT)
    process = subprocess.Popen(
        [str(PY), "-u", "gui.py", "--colab", "--noautoopen", "--port", str(GUI_PORT)],
        cwd=str(PROJECT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"gui.py exited with status {code}")
